## 0. Cài thư viện

In [ ]:
%pip install -q \
    pandas==2.2.2 numpy==1.26.4 tqdm==4.66.4 pydantic==2.7.4 python-dotenv==1.0.1 \
    neo4j==5.19.0 qdrant-client==1.9.1 sentence-transformers==3.0.1 torch==2.3.1 transformers==4.44.2 \
    langchain==0.2.6 langchain-openai==0.1.13 langchain-anthropic==0.1.20 \
    python-igraph==0.11.6 FlagEmbedding==1.2.11
print("✅ Dependencies installed")

: 

## 1. Imports, config và đường dẫn

In [22]:
from __future__ import annotations
import os, re, json, ast, math, hashlib, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Literal
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

REPO_ROOT = Path.cwd()
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(REPO_ROOT)))

GOOGLE_MAPS_PATH = Path(os.getenv("GOOGLE_MAPS_PATH", str(DATA_ROOT / "be_google_maps_unique.csv")))
FEEDBACK_PATH = Path(os.getenv("FEEDBACK_PATH", str(DATA_ROOT / "store_feedback_crawled.csv")))
FOODY_PATH = Path(os.getenv("FOODY_PATH", str(DATA_ROOT / "foody_hust_output" / "foody_hust_places_from_store_csv.csv")))

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", os.getenv("NEO4J_USERNAME", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
COLL_TEXT_UNIT = os.getenv("COLL_TEXT_UNIT", "graphrag_text_units_vietnamese")
COLL_RESTAURANT = os.getenv("COLL_RESTAURANT", "graphrag_restaurants_vietnamese")

# Vietnamese-first embedding. Override to VoVanPhuc/sup-SimCSE-VietNamese-phobert-base if desired.
EMBED_MODEL = os.getenv("EMBED_MODEL", "bkai-foundation-models/vietnamese-bi-encoder")
EMBED_PREFIX_QUERY = os.getenv("EMBED_PREFIX_QUERY", "")
EMBED_PREFIX_PASSAGE = os.getenv("EMBED_PREFIX_PASSAGE", "")

# PhoBERT-based sentiment model used as the classifier over aspect-specific review prompts.
# This notebook intentionally does not provide a fake classifier.
ASPECT_SENTIMENT_MODEL = os.getenv("ASPECT_SENTIMENT_MODEL", "wonrax/phobert-base-vietnamese-sentiment")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai").lower()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-20250514")

COMMUNITY_LEVEL = int(os.getenv("COMMUNITY_LEVEL", "0"))
RRF_K = int(os.getenv("RRF_K", "60"))
SIMILARITY_TOP_K = int(os.getenv("SIMILARITY_TOP_K", "8"))
SIMILARITY_MIN_SCORE = float(os.getenv("SIMILARITY_MIN_SCORE", "0.72"))

print("✅ Config loaded")
print("Repo root:", REPO_ROOT)
print("Data files:", GOOGLE_MAPS_PATH, FEEDBACK_PATH, FOODY_PATH, sep="\n  ")
print("Neo4j:", NEO4J_URI, "user=", NEO4J_USER)
print("Qdrant:", f"{QDRANT_HOST}:{QDRANT_PORT}")
print("Embedding:", EMBED_MODEL)
print("Aspect sentiment:", ASPECT_SENTIMENT_MODEL)

missing = [str(p) for p in [GOOGLE_MAPS_PATH, FEEDBACK_PATH, FOODY_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required data file(s):\n" + "\n".join(missing))


✅ Config loaded
Repo root: e:\restaurant-kg-recommender
Data files:
  e:\restaurant-kg-recommender\be_google_maps_unique.csv
  e:\restaurant-kg-recommender\store_feedback_crawled.csv
  e:\restaurant-kg-recommender\foody_hust_output\foody_hust_places_from_store_csv.csv
Neo4j: bolt://localhost:7687 user= neo4j
Qdrant: localhost:6333
Embedding: bkai-foundation-models/vietnamese-bi-encoder
Aspect sentiment: wonrax/phobert-base-vietnamese-sentiment


## 2. Inspect dữ liệu thật

In [4]:
raw_gmaps = pd.read_csv(GOOGLE_MAPS_PATH)
raw_feedback = pd.read_csv(FEEDBACK_PATH)
raw_foody = pd.read_csv(FOODY_PATH)

display(raw_gmaps.head(2))
display(raw_feedback.head(2))
display(raw_foody.head(2))

print("Shapes:")
print("  raw_gmaps   :", raw_gmaps.shape)
print("  raw_feedback:", raw_feedback.shape)
print("  raw_foody   :", raw_foody.shape)


,id,name,query_name,query_address,address,latitude,longitude,rating,review_count,price,...,phone,website,opening_hours,service_options,highlights,popular_for,offerings,atmosphere,crowd,dining_options
0,32441,Mì Cay SEOUL Trần Đại Nghĩa,Mì Cay Seoul - Trần Đại Nghĩa,"91 Trần Đại Nghĩa, Bách Khoa, Hai Bà Trưng, Hà...","223 Trần Đại Nghĩa, Đồng Tâm, Bạch Mai, Hà Nội...",20.998099,105.845996,2.4,266.0,1-100.000 ₫,...,+84 981 997 983,https://micayseoul.com/,"{""open_state"": ""Sắp đóng cửa · 22:00 · Mở cửa ...","{""Ăn_tại_chỗ"": true, ""Đồ_ăn_mang_đi"": true, ""g...",NaN,Bữa tối; Ăn tối một mình,Nhà hàng thức ăn nhanh,Ấm cúng; Thông thường,Nhóm; Sinh viên,Bữa trưa; Bữa tối; Dịch vụ ăn uống; Phục vụ tạ...
1,31947,Tuý food - mì trộn Indomie,Kufoodd - Mì Trộn Indomie & Đồ Ăn Nhanh - Đê T...,"22 Ngách 62B Ngõ 219 Đê Tô Hoàng, P. Cầu Dền, ...","Ngõ 25 Đ. Đại Cồ Việt, Cầu Dền, Bạch Mai, Hà N...",21.008567,105.850473,4.7,37.0,1-100.000 ₫,...,+84 979 599 902,NaN,"{""open_state"": ""Đang mở cửa · Đóng cửa vào 0:0...","{""Ăn_tại_chỗ"": true, ""Đồ_ăn_mang_đi"": true}",NaN,NaN,Nhà hàng thức ăn nhanh,Thông thường,Nhóm,Dịch vụ ăn uống; Phục vụ tại bàn


,store_id,store_name,rated_at,rating,feedback,crawl_status,error,source
0,32441,Mì Cay Seoul - Trần Đại Nghĩa,22:03 12/04/2026,4.0,đồ ngon mà mình gọi thêm trứng với phô mai thì...,ok,NaN,NaN
1,32441,Mì Cay Seoul - Trần Đại Nghĩa,19:49 12/04/2026,5.0,"Đáng đồng tiền, Sạch sẽ",ok,NaN,NaN


,input_store_id,input_store_name,input_address,matched_foody_url,crawl_status,error,restaurant_id,name,url,title_page,...,categories,cuisines,audiences,wifi,opening_hours,rating_quality,rating_position,rating_service,rating_price,rating_space
0,126162,Tiệm Trà Cô Chiêu - Milk Tea & Ăn Vặt - Lê Tha...,"76 Lê Thanh Nghị, Phường Bạch Mai, Hà Nội",NaN,foody_url_not_found,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,126679,Mì Cay Sassin 7 Cấp Độ Hàn Quốc - Bách Khoa,"104K9 Nguyễn Hiền, Phường Bạch Mai, Hà Nội",NaN,foody_url_not_found,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Shapes:
  raw_gmaps   : (49, 21)
  raw_feedback: (1068, 8)
  raw_foody   : (49, 37)


In [5]:
def _is_nan(x: Any) -> bool:
    return pd.isna(x)

def normalize_text(x: Any) -> str:
    if x is None or _is_nan(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKC", x)
    return re.sub(r"\s+", " ", x)

def slugify_vn(x: Any) -> str:
    x = normalize_text(x)
    x = ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn')
    x = x.replace("đ", "d")
    x = re.sub(r"[^a-z0-9]+", "-", x).strip("-")
    return x

def to_float(x: Any) -> Optional[float]:
    if x is None or _is_nan(x) or str(x).strip() == "":
        return None
    try:
        return float(x)
    except:
        s = str(x).replace(".", "").replace(",", ".")
        s = re.sub(r"[^0-9.\-]", "", s)
        try:
            return float(s)
        except:
            return None

def to_int(x: Any) -> Optional[int]:
    v = to_float(x)
    return None if v is None else int(v)

def parse_jsonish(x: Any) -> Any:
    if x is None or _is_nan(x):
        return None
    if isinstance(x, (list, dict)):
        return x
    s = str(x).strip()
    if not s:
        return None
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except:
            pass
    return s

def split_semi(x: Any) -> List[str]:
    if x is None or _is_nan(x):
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    parsed = parse_jsonish(x)
    if isinstance(parsed, list):
        return [str(i).strip() for i in parsed if str(i).strip()]
    s = str(x)
    parts = re.split(r"[;|,/]", s)
    return [p.strip() for p in parts if p and p.strip()]

def parse_price_band(text: Any) -> Optional[str]:
    if text is None or _is_nan(text):
        return None
    s = normalize_text(text)
    if not s:
        return None
    nums = re.findall(r"\d[\d\.]*", s)
    vals = []
    for n in nums:
        try:
            vals.append(int(float(n.replace(".", ""))))
        except:
            pass
    if not vals:
        return None
    hi = max(vals)
    if hi <= 50000:
        return "budget"
    if hi <= 120000:
        return "mid"
    return "premium"


## 3. Canonicalize schema và hợp nhất nguồn dữ liệu

In [6]:
def canonicalize_gmaps(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["store_id"] = df["id"].astype("Int64").astype(str)
    out["store_name"] = df["name"].fillna(df.get("query_name"))
    out["query_name"] = df.get("query_name")
    out["address"] = df["address"].fillna(df.get("query_address"))
    out["query_address"] = df.get("query_address")
    out["lat"] = df["latitude"].apply(to_float)
    out["lng"] = df["longitude"].apply(to_float)
    out["gmaps_rating"] = df["rating"].apply(to_float)
    out["gmaps_review_count"] = df["review_count"].apply(to_int)
    out["price_text"] = df.get("price")
    out["price_band"] = df.get("price", pd.Series([None]*len(df))).apply(parse_price_band)
    out["phone"] = df.get("phone")
    out["website"] = df.get("website")
    out["type_text"] = df.get("type")
    out["categories"] = df.get("type", pd.Series([None]*len(df))).apply(split_semi)
    out["opening_hours_raw"] = df.get("opening_hours")
    out["service_options_raw"] = df.get("service_options")
    out["atmosphere"] = df.get("atmosphere", pd.Series([None]*len(df))).apply(split_semi)
    out["crowd"] = df.get("crowd", pd.Series([None]*len(df))).apply(split_semi)
    out["popular_for"] = df.get("popular_for", pd.Series([None]*len(df))).apply(split_semi)
    out["dining_options"] = df.get("dining_options", pd.Series([None]*len(df))).apply(split_semi)
    out["offerings"] = df.get("offerings", pd.Series([None]*len(df))).apply(split_semi)
    out["store_key"] = out["store_id"].astype(str)
    out["name_norm"] = out["store_name"].apply(slugify_vn)
    out["addr_norm"] = out["address"].apply(slugify_vn)
    return out

def canonicalize_feedback(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "crawl_status" in df.columns:
        df = df[df["crawl_status"].fillna("ok").eq("ok")].copy()
    out = pd.DataFrame()
    out["store_id"] = df["store_id"].astype("Int64").astype(str)
    out["store_name"] = df["store_name"]
    out["rated_at"] = df.get("rated_at")
    out["rating"] = df["rating"].apply(to_float).fillna(3.0)
    out["feedback"] = df["feedback"].fillna("").astype(str)
    out["source"] = df.get("source", pd.Series(["delivery"]*len(df))).fillna("delivery")
    out["review_id"] = [
        hashlib.md5(f"{sid}|{rt}|{fb}".encode("utf-8")).hexdigest()[:16]
        for sid, rt, fb in zip(out["store_id"], out["rated_at"], out["feedback"])
    ]
    out["store_key"] = out["store_id"].astype(str)
    out["name_norm"] = out["store_name"].apply(slugify_vn)
    return out

def canonicalize_foody(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["input_store_id"] = df["input_store_id"].astype("Int64").astype(str)
    out["input_store_name"] = df["input_store_name"]
    out["crawl_status"] = df.get("crawl_status")
    out["foody_name"] = df.get("name")
    out["foody_address"] = df.get("address")
    out["district"] = df.get("district")
    out["area"] = df.get("area")
    out["city"] = df.get("city")
    out["foody_lat"] = df.get("lat").apply(to_float)
    out["foody_lng"] = df.get("lng").apply(to_float)
    out["foody_rating"] = df.get("avg_rating").apply(to_float)
    out["foody_review_count"] = df.get("total_review").apply(to_int)
    out["price_min"] = df.get("price_min").apply(to_float)
    out["price_max"] = df.get("price_max").apply(to_float)
    out["foody_price_band"] = df.get("price_max").apply(
        lambda x: "budget" if (to_float(x) or 0) <= 50000 else ("mid" if (to_float(x) or 0) <= 120000 else "premium")
        if to_float(x) is not None else None
    )
    out["categories"] = df.get("categories", pd.Series([None]*len(df))).apply(split_semi)
    out["cuisines"] = df.get("cuisines", pd.Series([None]*len(df))).apply(split_semi)
    out["audiences"] = df.get("audiences", pd.Series([None]*len(df))).apply(split_semi)
    out["opening_hours_foody"] = df.get("opening_hours")
    out["rating_quality"] = df.get("rating_quality").apply(to_float)
    out["rating_position"] = df.get("rating_position").apply(to_float)
    out["rating_service"] = df.get("rating_service").apply(to_float)
    out["rating_price"] = df.get("rating_price").apply(to_float)
    out["rating_space"] = df.get("rating_space").apply(to_float)
    out["store_key"] = out["input_store_id"].astype(str)
    out["name_norm"] = out["input_store_name"].apply(slugify_vn)
    return out

gmaps = canonicalize_gmaps(raw_gmaps)
feedback = canonicalize_feedback(raw_feedback)
foody = canonicalize_foody(raw_foody)


In [7]:
restaurants = gmaps.merge(
    foody.drop_duplicates("store_key"),
    on="store_key",
    how="left",
    suffixes=("", "_foody")
)

restaurants["name"] = restaurants["store_name"].fillna(restaurants["foody_name"]).fillna(restaurants["query_name"])
restaurants["address_final"] = restaurants["address"].fillna(restaurants["foody_address"]).fillna(restaurants["query_address"])
restaurants["district_final"] = restaurants["district"]
restaurants["city_final"] = restaurants["city"].fillna("Hà Nội")
restaurants["price_band_final"] = restaurants["price_band"].fillna(restaurants["foody_price_band"])

def merge_list_cols(a, b):
    aa = a if isinstance(a, list) else []
    bb = b if isinstance(b, list) else []
    out, seen = [], set()
    for x in aa + bb:
        x = str(x).strip()
        if x and x.lower() not in seen:
            out.append(x)
            seen.add(x.lower())
    return out

restaurants["categories_final"] = [merge_list_cols(a, b) for a, b in zip(restaurants["categories"], restaurants["categories_foody"])]
restaurants["cuisines_final"] = restaurants["cuisines"].apply(lambda x: x if isinstance(x, list) else [])
restaurants["audiences_final"] = [merge_list_cols(a, b) for a, b in zip(restaurants["crowd"], restaurants["audiences"])]

summary = pd.DataFrame({
    "store_key": restaurants["store_key"],
    "name": restaurants["name"],
    "address": restaurants["address_final"],
    "district": restaurants["district_final"],
    "city": restaurants["city_final"],
    "gmaps_rating": restaurants["gmaps_rating"],
    "foody_rating": restaurants["foody_rating"],
    "price_band": restaurants["price_band_final"],
    "categories": restaurants["categories_final"],
    "cuisines": restaurants["cuisines_final"],
    "atmosphere": restaurants["atmosphere"],
    "audiences": restaurants["audiences_final"],
    "lat": restaurants["lat"],
    "lng": restaurants["lng"],
})
display(summary.head(10))
print("✅ Unified restaurant table ready:", summary.shape)


,store_key,name,address,district,city,gmaps_rating,foody_rating,price_band,categories,cuisines,atmosphere,audiences,lat,lng
0,32441,Mì Cay SEOUL Trần Đại Nghĩa,"223 Trần Đại Nghĩa, Đồng Tâm, Bạch Mai, Hà Nội...",Quận Hai Bà Trưng,Hà Nội,2.4,6.554,mid,"[Nhà hàng Hàn Quốc, Nhà hàng]",[Món Hàn],"[Ấm cúng, Thông thường]","[Nhóm, Sinh viên]",20.998099,105.845996
1,31947,Tuý food - mì trộn Indomie,"Ngõ 25 Đ. Đại Cồ Việt, Cầu Dền, Bạch Mai, Hà N...",Quận Hai Bà Trưng,Hà Nội,4.7,0.000,mid,"[Nhà hàng, Quán ăn]",[],[Thông thường],[Nhóm],21.008567,105.850473
2,40268,Xuân Food - Mì Trộn Indomie - Minh Khai,"15 Ngõ 243 Ngõ 195 cũ, Minh Khai, Bạch Mai, Hà...",Quận Hai Bà Trưng,Hà Nội,4.7,0.000,mid,"[Nhà hàng ăn nhanh, Quán ăn]",[],[Thông thường],[Nhóm],20.996049,105.859160
3,72445,Bánh Mì Chảo - Bếp Cô Nụ,"Đại Học Kinh Tế Quốc Dân (NEU, 26 Ng. 183 P. T...",Quận Hai Bà Trưng,Hà Nội,4.7,0.000,mid,"[Nhà hàng Việt Nam, Quán ăn]",[],[],[],21.000230,105.846362
4,126679,Mỳ cay sasin,"119A Trần Đại Nghĩa, Bách Khoa, Bạch Mai, Hà N...",NaN,Hà Nội,3.9,NaN,mid,[Nhà hàng châu Á],[],"[Ấm cúng, Thông thường]","[Nhóm, Sinh viên]",21.002408,105.844889
5,80180,Quyên phở gà,"52 P. Vọng, Đồng Tâm, Bạch Mai, Hà Nội, Việt Nam",Quận Đống Đa,Hà Nội,NaN,0.000,mid,"[Nhà hàng phở, Quán ăn]",[],[Thông thường],[Nhóm],20.998194,105.842134
6,47053,Bee Quán,"2V22+HFM, Ng. Thái Lợi, Bạch Mai, Hà Nội, Việt...",Quáº­n Hai BÃ TrÆ°ng,HÃ Ná»™i,4.3,9.858,mid,"[Quán ăn nhỏ, QuÃ¡n Äƒn]",[],[Thông thường],[],21.001455,105.851172
7,130783,Bún bò Huế Anh Quân,"51a Đ. Đại Cồ Việt, Lê Đại Hành, Bạch Mai, Hà ...",Quận Hai Bà Trưng,Hà Nội,4.2,0.000,mid,"[Nhà hàng, Ăn vặt, vỉa hè]",[],[Thông thường],[Nhóm],21.007888,105.846907
8,86930,Bún Cá Cô Minh,"3 Ngh. 40/2 P. Tạ Quang Bửu, Bách Khoa, Bạch...",Quận Hai Bà Trưng,Hà Nội,4.2,0.000,budget,"[Nhà hàng ăn nhanh, Quán ăn]","[Món Bắc, Món Miền Trung, Món Miền Nam, Tây Ng...",[Thông thường],"[Sinh viên, Cặp đôi, Gia đình, Nhóm hội, Giới ...",21.004094,105.846557
9,117526,Bếp Béo - Chuẩn Vị Trung Hoa,"68 P. Ngụy Như Kon Tum, Nhân Chính, Thanh Xuân...",Quận Hai Bà Trưng,Hà Nội,4.6,0.000,budget,"[Nhà hàng Trung Quốc có giao hàng, Quán ăn]",[],[],[],20.998808,105.800613


✅ Unified restaurant table ready: (49, 14)


## 3.1. LLM-based entity extraction (món ăn, địa điểm con)

In [ ]:
# FIX (Vấn đề 2): LLM-based Named Entity Extraction cho domain restaurant.
# GraphRAG chuẩn extract Named Entities từ TextUnit để tạo Entity nodes riêng.
# Ở đây ta extract dish_name từ reviews để enable "quán nào có bún bò ngon nhất" queries.

class DishEntity(BaseModel):
    dish_name: str
    mention_count: int = 1
    sentiment_hint: Optional[Literal["positive", "negative", "neutral"]] = None

class EntityExtractionResult(BaseModel):
    dishes: List[DishEntity] = Field(default_factory=list)
    location_mentions: List[str] = Field(default_factory=list)  # street/area names

ENTITY_SYSTEM = """Bạn là NER model cho domain nhà hàng Việt Nam.
Trích xuất từ review:
1. dishes: tên các món ăn được nhắc đến (normalize về dạng chuẩn, ví dụ "bún bò Huế" không phải "bún bò")
2. location_mentions: tên đường, khu vực, tòa nhà được nhắc đến

Chỉ trích xuất những gì có trong text. Trả về JSON theo schema EntityExtractionResult.
Nếu không có entity nào, trả về {"dishes": [], "location_mentions": []}.
"""
entity_prompt = ChatPromptTemplate.from_messages([
    ("system", ENTITY_SYSTEM),
    ("human", "Review: {review}"),
])
entity_llm = llm.with_structured_output(EntityExtractionResult)


def extract_entities_batch(
    reviews: List[str],
    store_keys: List[str],
    sample_rate: float = 0.3,
    max_reviews: int = 500,
) -> pd.DataFrame:
    """Extract dish entities từ reviews (sampling để giảm LLM cost).

    sample_rate: tỷ lệ review sẽ được extract (default 30%)
    max_reviews: hard cap số reviews tối đa
    """
    import random
    combined = list(zip(store_keys, reviews))
    random.seed(42)
    if sample_rate < 1.0:
        n = min(int(len(combined) * sample_rate), max_reviews)
        combined = random.sample(combined, n)

    rows = []
    for store_key, review in tqdm(combined, desc="Entity extraction"):
        if not review or len(review.strip()) < 10:
            continue
        try:
            result = (entity_prompt | entity_llm).invoke({"review": review[:500]})
            for dish in result.dishes:
                rows.append({
                    "store_key": store_key,
                    "dish_name": dish.dish_name.strip().lower(),
                    "mention_count": dish.mention_count,
                    "sentiment_hint": dish.sentiment_hint,
                })
        except Exception:
            pass

    if not rows:
        return pd.DataFrame(columns=["store_key", "dish_name", "mention_count", "sentiment_hint"])

    df = pd.DataFrame(rows)
    # Aggregate: count total mentions per (store_key, dish_name)
    agg = df.groupby(["store_key", "dish_name"]).agg(
        total_mentions=("mention_count", "sum"),
        positive_count=("sentiment_hint", lambda x: (x == "positive").sum()),
    ).reset_index()
    return agg


def upsert_dish_entities(client: Neo4jClient, dish_df: pd.DataFrame):
    """Write DishEntity nodes and (Restaurant)-[:SERVES]->(DishEntity) edges."""
    if dish_df.empty:
        print("⚠️  No dish entities to upsert")
        return
    rows = dish_df.rename(columns={"dish_name": "name"}).to_dict("records")
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (d:DishEntity {name: row.name})
    MERGE (r)-[s:SERVES]->(d)
    SET s.mention_count = row.total_mentions,
        s.positive_mentions = row.positive_count,
        s.updated_at = datetime()
    """, {"rows": rows})
    print(f"✅ Upserted {len(dish_df)} dish entity edges")


# Run entity extraction (sampling 30% of reviews to control LLM cost)
# Set sample_rate=1.0 for full extraction at indexing time
dish_entities = extract_entities_batch(
    reviews=feedback_proc["feedback_norm"].tolist(),
    store_keys=feedback_proc["store_key"].tolist(),
    sample_rate=0.3,
    max_reviews=500,
)
print(f"Top dishes found:\n{dish_entities.nlargest(10, 'total_mentions')[['store_key','dish_name','total_mentions']].to_string()}")

## 4. PhoBERT-based aspect sentiment và TextUnit evidence


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ASPECTS: Dict[str, str] = {
    "food_quality": "chất lượng món ăn, độ ngon, hương vị, độ tươi",
    "service": "thái độ và chất lượng phục vụ của nhân viên",
    "cleanliness": "vệ sinh, sạch sẽ, mùi, an toàn thực phẩm",
    "packaging": "đóng gói, giao hàng không đổ vỡ, đầy đủ món",
    "price": "giá cả, độ đáng tiền, phù hợp sinh viên",
    "space": "không gian quán, độ rộng, yên tĩnh, thoải mái",
    "speed": "tốc độ phục vụ, thời gian chờ món hoặc giao hàng",
}

LABEL_TO_SCORE = {
    "negative": -1.0, "neg": -1.0, "0": -1.0,
    "neutral": 0.0, "neu": 0.0, "1": 0.0,
    "positive": 1.0, "pos": 1.0, "2": 1.0,
}

def normalize_review_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s.lower())
    return re.sub(r"\s+", " ", s).strip()

class PhoBERTAspectSentiment:
    """Aspect sentiment with batched multi-aspect inference.

    FIX (Vấn đề 6): Thay vì gọi model 7 lần per review (7 aspects × N reviews = 7N calls),
    ta build 7×N texts trước, rồi chạy batched inference 1 lần dùng tokenizer padding.
    Giảm 5–10× thời gian indexing so với sequential forward pass.
    """
    def __init__(self, model_name: str, device: Optional[str] = None, batch_size: int = 64):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.id2label = {int(k): str(v).lower() for k, v in self.model.config.id2label.items()}
        if not self.id2label:
            raise RuntimeError(f"Model {model_name} does not expose id2label; cannot map sentiment labels safely.")
        self.aspect_names = list(ASPECTS.keys())
        self.aspect_descs = list(ASPECTS.values())

    @torch.inference_mode()
    def _batch_infer(self, texts: List[str]) -> List[float]:
        """Run batched inference, return expected sentiment score per text."""
        scores = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self.tokenizer(
                batch,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt",
            ).to(self.device)
            logits = self.model(**enc).logits  # (B, n_labels)
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
            for row in probs:
                expected = 0.0
                mass = 0.0
                for j, p in enumerate(row):
                    label = self.id2label.get(j, str(j)).lower()
                    score = LABEL_TO_SCORE.get(label)
                    if score is None:
                        raise RuntimeError(f"Unsupported label {label} from {self.model_name}. Update LABEL_TO_SCORE.")
                    expected += float(p) * score
                    mass += float(p)
                scores.append(round(expected / max(mass, 1e-9), 4))
        return scores

    def score_reviews_batch(self, reviews: List[str]) -> List[Dict[str, float]]:
        """Score all aspects for all reviews in one batched forward pass sequence.

        Build (n_aspects × n_reviews) texts, infer in batches, reshape.
        This is O(n_aspects * ceil(N/batch)) batches instead of O(n_aspects * N) sequential calls.
        """
        aspect_texts = []
        for asp_desc in self.aspect_descs:
            for review in reviews:
                aspect_texts.append(f"Khía cạnh: {asp_desc}. Nhận xét: {review}")

        all_scores = self._batch_infer(aspect_texts)

        n = len(reviews)
        results = []
        for review_idx in range(n):
            result = {}
            for asp_idx, asp_name in enumerate(self.aspect_names):
                result[asp_name] = all_scores[asp_idx * n + review_idx]
            results.append(result)
        return results

    # backward compat: single review
    def score_review(self, review: str) -> Dict[str, float]:
        return self.score_reviews_batch([review])[0]

aspect_model = PhoBERTAspectSentiment(ASPECT_SENTIMENT_MODEL)

def classify_sentiment_from_aspects(aspect_scores: Dict[str, float]) -> str:
    avg = float(np.mean(list(aspect_scores.values()))) if aspect_scores else 0.0
    if avg >= 0.20:
        return "positive"
    if avg <= -0.20:
        return "negative"
    return "neutral"

feedback_proc = feedback.copy()
feedback_proc["feedback_norm"] = feedback_proc["feedback"].apply(normalize_review_text)

# Batched scoring: 1 call per aspect × ceil(N/batch) instead of 7 calls per review
print("Running batched PhoBERT aspect inference...")
reviews_list = feedback_proc["feedback_norm"].tolist()
aspect_scores_list = aspect_model.score_reviews_batch(reviews_list)
feedback_proc["aspect_scores"] = aspect_scores_list
feedback_proc["sentiment"] = feedback_proc["aspect_scores"].apply(classify_sentiment_from_aspects)
print(f"✅ PhoBERT batched inference done: {len(feedback_proc)} reviews")
feedback_proc.head(3)

tokenizer_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

c:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--wonrax--phobert-base-vietnamese-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

...

c:\Users\Admin\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

PhoBERT aspect sentiment:   0%|          | 0/1055 [00:00<?, ?it/s]

,store_id,store_name,rated_at,rating,feedback,source,review_id,store_key,name_norm,feedback_norm,aspect_scores,sentiment
0,32441,Mì Cay Seoul - Trần Đại Nghĩa,22:03 12/04/2026,4.0,đồ ngon mà mình gọi thêm trứng với phô mai thì...,delivery,828c34dc79189f9e,32441,mi-cay-seoul-tran-dai-nghia,đồ ngon mà mình gọi thêm trứng với phô mai thì...,"{'food_quality': 0.9598, 'service': -0.6397, '...",positive
1,32441,Mì Cay Seoul - Trần Đại Nghĩa,19:49 12/04/2026,5.0,"Đáng đồng tiền, Sạch sẽ",delivery,c92f5b5b335e1223,32441,mi-cay-seoul-tran-dai-nghia,"đáng đồng tiền, sạch sẽ","{'food_quality': 0.9911, 'service': 0.9912, 'c...",positive
2,32441,Mì Cay Seoul - Trần Đại Nghĩa,15:22 05/04/2026,5.0,"Ngon xỉu, Đáng đồng tiền, Đóng gói tốt, Sạch sẽ",delivery,0286508b80ab3827,32441,mi-cay-seoul-tran-dai-nghia,"ngon xỉu, đáng đồng tiền, đóng gói tốt, sạch sẽ","{'food_quality': 0.9911, 'service': 0.9911, 'c...",positive


In [9]:
attr_rows = []
for store_key, grp in feedback_proc.groupby("store_key"):
    bucket = defaultdict(list)
    for asp_map in grp["aspect_scores"]:
        for k, v in asp_map.items():
            bucket[k].append(v)
    for aspect, vals in bucket.items():
        attr_rows.append({
            "store_key": store_key,
            "attribute_type": aspect,
            "attribute_score": round(float(np.mean(vals)), 3),
            "sample_count": len(vals)
        })
restaurant_attrs = pd.DataFrame(attr_rows)
display(restaurant_attrs.head(10))


,store_key,attribute_type,attribute_score,sample_count
0,103271,food_quality,0.877,29
1,103271,service,0.770,29
2,103271,cleanliness,0.936,29
3,103271,packaging,0.664,29
4,103271,price,0.922,29
5,103271,space,0.949,29
6,103271,speed,0.292,29
7,103957,food_quality,0.980,8
8,103957,service,0.807,8
9,103957,cleanliness,0.986,8


In [10]:
def build_text_unit_text(row: pd.Series, restaurant_name: str = "") -> str:
    aspects = ", ".join(f"{k}={v:+.2f}" for k, v in (row["aspect_scores"] or {}).items())
    return (
        f"Tên quán: {restaurant_name or row['store_name']}\n"
        f"Nguồn: {row['source']}\n"
        f"Rating người dùng: {row['rating']}\n"
        f"Sentiment tổng hợp: {row['sentiment']}\n"
        f"Aspect sentiment: {aspects}\n"
        f"Nội dung review: {row['feedback']}"
    )

name_map = summary.set_index("store_key")["name"].to_dict()
text_units = feedback_proc.copy()
text_units["text_unit_id"] = "tu_" + text_units["review_id"].astype(str)
text_units["chunk_text"] = [build_text_unit_text(r, name_map.get(r["store_key"], r["store_name"])) for _, r in text_units.iterrows()]
review_chunks = text_units.copy()  # backward-compatible alias for functions below
text_units[["store_key", "text_unit_id", "chunk_text"]].head(3)


,store_key,text_unit_id,chunk_text
0,32441,tu_828c34dc79189f9e,Tên quán: Mì Cay SEOUL Trần Đại Nghĩa\nNguồn: ...
1,32441,tu_c92f5b5b335e1223,Tên quán: Mì Cay SEOUL Trần Đại Nghĩa\nNguồn: ...
2,32441,tu_0286508b80ab3827,Tên quán: Mì Cay SEOUL Trần Đại Nghĩa\nNguồn: ...


## 4.1. Text chunking chuẩn GraphRAG (sliding window over long reviews)

In [ ]:
def chunk_review_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80,
) -> List[str]:
    """FIX (Vấn đề 1): Proper sliding window text chunking.

    Thay vì review = 1 TextUnit, chia review dài thành nhiều chunk:
    - chunk_size ~400 ký tự ≈ 300-500 token Vietnamese
    - overlap 80 ký tự để giữ context continuity giữa các chunk
    - Review ngắn (< chunk_size): vẫn ra 1 chunk, không thay đổi behavior
    """
    text = text.strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end >= len(text):
            chunks.append(text[start:])
            break
        # Ưu tiên cắt tại ranh giới câu
        boundary = max(
            text.rfind(".", start, end),
            text.rfind("!", start, end),
            text.rfind("?", start, end),
            text.rfind("\n", start, end),
        )
        if boundary > start + overlap:
            end = boundary + 1
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if c.strip()]


def build_text_units_with_chunking(
    feedback_proc_df: pd.DataFrame,
    name_map: Dict[str, str],
    chunk_size: int = 400,
    overlap: int = 80,
) -> pd.DataFrame:
    """Build TextUnit DataFrame với proper sliding window chunking.

    Mỗi review dài có thể tạo ra nhiều TextUnit chunk.
    text_unit_id = review_id + "_chunk{i}" cho multi-chunk reviews.
    """
    rows = []
    for _, r in feedback_proc_df.iterrows():
        review_text = str(r["feedback"])
        aspect_str = ", ".join(
            f"{k}={v:+.2f}" for k, v in (r["aspect_scores"] or {}).items()
        )
        header = (
            f"Tên quán: {name_map.get(r['store_key'], r['store_name'])}\n"
            f"Nguồn: {r['source']}\n"
            f"Rating người dùng: {r['rating']}\n"
            f"Sentiment tổng hợp: {r['sentiment']}\n"
            f"Aspect sentiment: {aspect_str}\n"
            f"Nội dung review: "
        )
        chunks = chunk_review_text(review_text, chunk_size=chunk_size, overlap=overlap)
        if not chunks:
            chunks = ["(empty review)"]

        for i, chunk in enumerate(chunks):
            chunk_id = f"{r['review_id']}_{i}" if len(chunks) > 1 else r["review_id"]
            rows.append({
                "text_unit_id": "tu_" + chunk_id,
                "review_id": r["review_id"],
                "store_key": r["store_key"],
                "store_name": r["store_name"],
                "rating": r["rating"],
                "sentiment": r["sentiment"],
                "aspect_scores": r["aspect_scores"],
                "source": r["source"],
                "feedback": review_text,
                "chunk_text": header + chunk,
                "chunk_index": i,
                "n_chunks": len(chunks),
            })
    return pd.DataFrame(rows)


# Rebuild text_units with proper chunking (replaces flat 1-review-1-unit)
text_units = build_text_units_with_chunking(feedback_proc, name_map)
review_chunks = text_units.copy()
multi_chunk = (text_units["n_chunks"] > 1).sum()
print(f"✅ TextUnits after chunking: {len(text_units)} chunks from {len(feedback_proc)} reviews")
print(f"   Reviews producing multiple chunks: {multi_chunk}")
text_units[["store_key", "text_unit_id", "chunk_index", "n_chunks"]].head(5)

## 5. Graph schema sâu hơn

In [11]:
GRAPH_SCHEMA_NOTE = """
GraphRAG-style schema used in this notebook

Core indexing nodes:
- (:Restaurant)        domain entity
- (:Review)            raw source record
- (:TextUnit)          chunk/text unit; each review is represented as one text unit
- (:Attribute)         aspect sentiment aggregate per restaurant
- (:Community)         detected cluster over the restaurant similarity graph
- (:CommunityReport)   LLM-written summary of a community for global/contextual retrieval

Domain/context nodes:
- (:Area), (:Cuisine), (:Category), (:PriceBand), (:AtmosphereTag)
- (:DishEntity)           named dish/food entity extracted from reviews (LLM NER)

Core relations:
- (Review)-[:HAS_TEXT_UNIT]->(TextUnit)
- (TextUnit)-[:ABOUT]->(Restaurant)
- (TextUnit)-[:MENTIONS_ASPECT]->(Attribute)
- (Restaurant)-[:HAS_ATTRIBUTE]->(Attribute)
- (Restaurant)-[:IN_COMMUNITY]->(Community)
- (Community)-[:HAS_REPORT]->(CommunityReport)
- (Restaurant)-[:SIMILAR_TO {similarity, method:'embedding_cosine'}]-(Restaurant)
- (Restaurant)-[:SERVES {mention_count, positive_mentions}]->(DishEntity)
"""
print(GRAPH_SCHEMA_NOTE)



GraphRAG-style schema used in this notebook

Core indexing nodes:
- (:Restaurant)        domain entity
- (:Review)            raw source record
- (:TextUnit)          chunk/text unit; each review is represented as one text unit
- (:Attribute)         aspect sentiment aggregate per restaurant
- (:Community)         detected cluster over the restaurant similarity graph
- (:CommunityReport)   LLM-written summary of a community for global/contextual retrieval

Domain/context nodes:
- (:Area), (:Cuisine), (:Category), (:PriceBand), (:AtmosphereTag)

Core relations:
- (Review)-[:HAS_TEXT_UNIT]->(TextUnit)
- (TextUnit)-[:ABOUT]->(Restaurant)
- (TextUnit)-[:MENTIONS_ASPECT]->(Attribute)
- (Restaurant)-[:HAS_ATTRIBUTE]->(Attribute)
- (Restaurant)-[:IN_COMMUNITY]->(Community)
- (Community)-[:HAS_REPORT]->(CommunityReport)
- (Restaurant)-[:SIMILAR_TO {similarity, method:'embedding_cosine'}]-(Restaurant)



In [15]:
with neo4j_client.driver.session() as session:
    rows = session.run("""
    SHOW INDEXES
    YIELD name, type, entityType, labelsOrTypes, properties
    WHERE labelsOrTypes = ['Attribute']
    RETURN name, type, entityType, labelsOrTypes, properties
    """).data()

rows

[{'name': 'attr_store_type',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Attribute'],
  'properties': ['store_key', 'type']},
 {'name': 'attr_type',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Attribute'],
  'properties': ['type']}]

In [17]:
with neo4j_client.driver.session() as session:
    session.run("DROP INDEX attr_type IF EXISTS")

In [24]:
from neo4j import GraphDatabase

class Neo4jClient:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()

    def close(self):
        self.driver.close()

    def run(self, query: str, params: dict | None = None):
        with self.driver.session() as s:
            return [r.data() for r in s.run(query, params or {})]

    def create_schema(self):
        stmts = [
            # Drop old Enterprise-only / conflicting schema if it exists
            "DROP CONSTRAINT attr_key IF EXISTS",
            "DROP CONSTRAINT area_key IF EXISTS",

            # Community-compatible constraints
            "CREATE CONSTRAINT restaurant_key IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.store_key IS UNIQUE",
            "CREATE CONSTRAINT review_key IF NOT EXISTS FOR (r:Review) REQUIRE r.review_id IS UNIQUE",
            "CREATE CONSTRAINT text_unit_key IF NOT EXISTS FOR (t:TextUnit) REQUIRE t.text_unit_id IS UNIQUE",

            # Neo4j Community does NOT support NODE KEY.
            # Use synthetic unique ids instead.
            "CREATE CONSTRAINT attr_id_key IF NOT EXISTS FOR (a:Attribute) REQUIRE a.attribute_id IS UNIQUE",
            "CREATE CONSTRAINT area_id_key IF NOT EXISTS FOR (a:Area) REQUIRE a.area_id IS UNIQUE",

            "CREATE CONSTRAINT cuisine_name IF NOT EXISTS FOR (c:Cuisine) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT priceband_name IF NOT EXISTS FOR (p:PriceBand) REQUIRE p.name IS UNIQUE",
            "CREATE CONSTRAINT atmos_name IF NOT EXISTS FOR (a:AtmosphereTag) REQUIRE a.name IS UNIQUE",
            "CREATE CONSTRAINT community_key IF NOT EXISTS FOR (c:Community) REQUIRE c.community_id IS UNIQUE",
            "CREATE CONSTRAINT community_report_key IF NOT EXISTS FOR (cr:CommunityReport) REQUIRE cr.report_id IS UNIQUE",

            "CREATE INDEX rest_rating IF NOT EXISTS FOR (r:Restaurant) ON (r.gmaps_rating)",
            "CREATE INDEX text_unit_store IF NOT EXISTS FOR (t:TextUnit) ON (t.store_key)",
            "CREATE INDEX attr_type IF NOT EXISTS FOR (a:Attribute) ON (a.type)",
            "CREATE INDEX community_level IF NOT EXISTS FOR (c:Community) ON (c.level)",
        ]

        for q in stmts:
            self.run(q)


neo4j_client = Neo4jClient(
    NEO4J_URI,
    NEO4J_USER,
    NEO4J_PASSWORD,
)

neo4j_client.create_schema()
print("✅ Neo4j connected and schema is ready")

✅ Neo4j connected and schema is ready


In [26]:
def upsert_restaurants_graph(client: Neo4jClient, restaurants_df: pd.DataFrame):
    rows = []
    for _, r in restaurants_df.iterrows():
        district = r["district"]
        city = r["city"]

        rows.append({
            "store_key": r["store_key"],
            "name": r["name"],
            "address": r["address"],
            "district": district,
            "city": city,
            "area_id": f"{city}:{district}" if pd.notna(city) and pd.notna(district) else None,
            "lat": r["lat"],
            "lng": r["lng"],
            "gmaps_rating": r["gmaps_rating"],
            "foody_rating": r["foody_rating"],
            "price_band": r["price_band"],
            "categories": r["categories"],
            "cuisines": r["cuisines"],
            "atmosphere": r["atmosphere"],
            "audiences": r["audiences"],
        })

    client.run("""
    UNWIND $rows AS row
    MERGE (r:Restaurant {store_key: row.store_key})
    SET r.name = row.name,
        r.address = row.address,
        r.district = row.district,
        r.city = row.city,
        r.lat = row.lat,
        r.lng = row.lng,
        r.gmaps_rating = row.gmaps_rating,
        r.foody_rating = row.foody_rating,
        r.audiences = row.audiences,
        r.updated_at = datetime()

    WITH r, row
    FOREACH (cat IN coalesce(row.categories, []) |
        MERGE (c:Category {name: cat})
        MERGE (r)-[:HAS_CATEGORY]->(c)
    )

    FOREACH (cui IN coalesce(row.cuisines, []) |
        MERGE (c:Cuisine {name: cui})
        MERGE (r)-[:HAS_CUISINE]->(c)
    )

    FOREACH (atm IN coalesce(row.atmosphere, []) |
        MERGE (a:AtmosphereTag {name: atm})
        MERGE (r)-[:HAS_ATMOSPHERE]->(a)
    )

    FOREACH (_ IN CASE WHEN row.price_band IS NULL THEN [] ELSE [1] END |
        MERGE (p:PriceBand {name: row.price_band})
        MERGE (r)-[:HAS_PRICE_BAND]->(p)
    )

    FOREACH (_ IN CASE WHEN row.area_id IS NULL THEN [] ELSE [1] END |
        MERGE (a:Area {area_id: row.area_id})
        SET a.name = row.district,
            a.city = row.city
        MERGE (r)-[:IN_AREA]->(a)
    )
    """, {"rows": rows})


def upsert_attributes_graph(client: Neo4jClient, attrs_df: pd.DataFrame):
    rows = []
    for _, r in attrs_df.iterrows():
        store_key = r["store_key"]
        attr_type = r["attribute_type"]

        rows.append({
            "attribute_id": f"{store_key}:{attr_type}",
            "store_key": store_key,
            "attribute_type": attr_type,
            "attribute_score": r["attribute_score"],
            "sample_count": r["sample_count"],
        })

    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (a:Attribute {attribute_id: row.attribute_id})
    SET a.store_key = row.store_key,
        a.type = row.attribute_type,
        a.score = row.attribute_score,
        a.sample_count = row.sample_count,
        a.updated_at = datetime()
    MERGE (r)-[:HAS_ATTRIBUTE]->(a)
    """, {"rows": rows})


def upsert_reviews_and_text_units_graph(client: Neo4jClient, units_df: pd.DataFrame):
    rows = []
    for _, r in units_df.iterrows():
        rows.append({
            "review_id": r["review_id"],
            "text_unit_id": r["text_unit_id"],
            "store_key": r["store_key"],
            "feedback": r["feedback"],
            "chunk_text": r["chunk_text"],
            "rating": r["rating"],
            "rated_at": str(r["rated_at"]),
            "sentiment": r["sentiment"],
            "aspect_scores": json.dumps(r["aspect_scores"], ensure_ascii=False),
            "source": r["source"],
        })

    client.run("""
    UNWIND $rows AS row
    MATCH (rest:Restaurant {store_key: row.store_key})

    MERGE (rv:Review {review_id: row.review_id})
    SET rv.feedback = row.feedback,
        rv.rating = row.rating,
        rv.rated_at = row.rated_at,
        rv.sentiment = row.sentiment,
        rv.aspect_scores = row.aspect_scores,
        rv.source = row.source

    MERGE (tu:TextUnit {text_unit_id: row.text_unit_id})
    SET tu.text = row.chunk_text,
        tu.store_key = row.store_key,
        tu.source = row.source,
        tu.review_id = row.review_id,
        tu.sentiment = row.sentiment,
        tu.rating = row.rating,
        tu.updated_at = datetime()

    MERGE (rv)-[:HAS_TEXT_UNIT]->(tu)
    MERGE (tu)-[:ABOUT]->(rest)

    WITH tu, row
    MATCH (att:Attribute {store_key: row.store_key})
    MERGE (tu)-[:MENTIONS_ASPECT]->(att)
    """, {"rows": rows})


upsert_restaurants_graph(neo4j_client, summary)
upsert_attributes_graph(neo4j_client, restaurant_attrs)
upsert_reviews_and_text_units_graph(neo4j_client, text_units)

print("✅ Graph upsert complete: Restaurant, Review, TextUnit, Attribute and domain nodes")

# Upsert dish entities extracted in Section 3.1
if 'dish_entities' in dir() and not dish_entities.empty:
    upsert_dish_entities(neo4j_client, dish_entities)
else:
    print('⚠️  dish_entities not found — run Section 3.1 first')

✅ Graph upsert complete: Restaurant, Review, TextUnit, Attribute and domain nodes


## 6. Embedding similarity edges + Leiden communities


In [ ]:
# Similarity edges and communities are built after embeddings are created in Section 7.1.


## 7. Vietnamese embeddings và Qdrant indexing


In [1]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Loading Vietnamese embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = len(embed_model.encode("kiểm tra chiều embedding", normalize_embeddings=True))
print("✅ Embedding dim:", EMBED_DIM)

def _embed(text: str, prefix: str = "") -> List[float]:
    return embed_model.encode(prefix + text, normalize_embeddings=True).tolist()

def emb_passage(text: str) -> List[float]:
    return _embed(text, EMBED_PREFIX_PASSAGE)

def emb_query(text: str) -> List[float]:
    return _embed(text, EMBED_PREFIX_QUERY)

qdrant = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
qdrant.get_collections()  # fail fast if the service is unavailable
print("✅ Qdrant connected")

def recreate_collection(client: QdrantClient, name: str, dim: int):
    existing = {c.name for c in client.get_collections().collections}
    if name in existing:
        client.delete_collection(name)
    client.create_collection(name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    print("Recreated:", name)

recreate_collection(qdrant, COLL_TEXT_UNIT, EMBED_DIM)
recreate_collection(qdrant, COLL_RESTAURANT, EMBED_DIM)


KeyboardInterrupt: 

In [ ]:
def build_restaurant_summary_doc(store_key: str) -> str:
    row = summary.set_index("store_key").loc[store_key]
    attrs = restaurant_attrs[restaurant_attrs["store_key"].eq(store_key)]
    attr_text = ", ".join(f"{r.attribute_type}={r.attribute_score:+.2f}" for _, r in attrs.sort_values("attribute_type").iterrows())
    return "\n".join([
        f"Tên quán: {row['name']}",
        f"Địa chỉ: {row['address']}",
        f"Khu vực: {row['district']}, {row['city']}",
        f"Rating Google Maps: {row['gmaps_rating']}",
        f"Foody rating: {row['foody_rating']}",
        f"Price band: {row['price_band']}",
        f"Categories: {', '.join(row['categories'] or [])}",
        f"Cuisines: {', '.join(row['cuisines'] or [])}",
        f"Atmosphere: {', '.join(row['atmosphere'] or [])}",
        f"Audience: {', '.join(row['audiences'] or [])}",
        f"Aggregated aspect sentiment: {attr_text}",
    ])

restaurant_docs = pd.DataFrame({
    "store_key": summary["store_key"],
    "doc_text": summary["store_key"].apply(build_restaurant_summary_doc),
})
restaurant_docs["embedding"] = [emb_passage(x) for x in tqdm(restaurant_docs["doc_text"], desc="Embed restaurants")]
text_units["embedding"] = [emb_passage(x) for x in tqdm(text_units["chunk_text"], desc="Embed text units")]

def stable_int_id(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

def index_restaurant_docs():
    points = []
    meta_by_key = summary.set_index("store_key")
    for _, row in restaurant_docs.iterrows():
        meta = meta_by_key.loc[row["store_key"]]
        points.append(PointStruct(
            id=stable_int_id("rest-" + row["store_key"]),
            vector=row["embedding"],
            payload={
                "store_key": row["store_key"], "name": meta["name"], "address": meta["address"],
                "district": meta["district"], "city": meta["city"], "rating": meta["gmaps_rating"],
                "price_band": meta["price_band"], "doc_text": row["doc_text"], "doc_type": "restaurant_summary",
            }
        ))
    qdrant.upsert(collection_name=COLL_RESTAURANT, points=points)
    print(f"✅ Indexed restaurant docs: {len(points)}")

def index_text_units():
    points = []
    for _, row in text_units.iterrows():
        points.append(PointStruct(
            id=stable_int_id("tu-" + row["text_unit_id"]),
            vector=row["embedding"],
            payload={
                "text_unit_id": row["text_unit_id"], "review_id": row["review_id"], "store_key": row["store_key"],
                "store_name": name_map.get(row["store_key"], row["store_name"]), "rating": row["rating"],
                "sentiment": row["sentiment"], "feedback": row["feedback"], "aspect_scores": row["aspect_scores"],
                "doc_text": row["chunk_text"], "doc_type": "text_unit",
            }
        ))
    qdrant.upsert(collection_name=COLL_TEXT_UNIT, points=points)
    print(f"✅ Indexed text units: {len(points)}")

index_restaurant_docs()
index_text_units()


## 7.1. Embedding similarity edges + Leiden communities


In [ ]:
def cosine_np(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9))

def build_similarity_edges_from_embeddings(client: Neo4jClient, docs_df: pd.DataFrame, top_k: int = SIMILARITY_TOP_K, min_score: float = SIMILARITY_MIN_SCORE):
    keys = docs_df["store_key"].tolist()
    mat = np.asarray(docs_df["embedding"].tolist(), dtype=np.float32)
    mat = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9)
    sim = mat @ mat.T
    edges = []
    for i, a in enumerate(keys):
        order = np.argsort(-sim[i])
        kept = 0
        for j in order:
            if i == j or i > j:
                continue
            score = float(sim[i, j])
            if score < min_score:
                continue
            edges.append({"a": a, "b": keys[j], "sim": round(score, 4)})
            kept += 1
            if kept >= top_k:
                break
    client.run("MATCH (:Restaurant)-[s:SIMILAR_TO]-(:Restaurant) DELETE s")
    if edges:
        client.run("""
        UNWIND $edges AS e
        MATCH (a:Restaurant {store_key: e.a})
        MATCH (b:Restaurant {store_key: e.b})
        MERGE (a)-[s:SIMILAR_TO]-(b)
        SET s.similarity = e.sim, s.method = 'embedding_cosine', s.updated_at = datetime()
        """, {"edges": edges})
    return len(edges), edges


def build_communities_igraph(
    client: Neo4jClient,
    keys: List[str],
    edges: List[dict],
    level: int = COMMUNITY_LEVEL,
) -> List[dict]:
    """FIX (Vấn đề 3): Leiden community detection bằng python-igraph, không cần Neo4j GDS.

    Thay thế hoàn toàn gds.leiden.write:
    - Dùng igraph.Graph để xây undirected weighted graph từ similarity edges
    - Chạy community_leiden() với objective='modularity' (tương đương GDS Leiden)
    - Ghi community_id trực tiếp lên Neo4j bằng Cypher UNWIND batch
    Tăng portability: chạy trên Neo4j Community Edition (miễn phí).
    """
    import igraph as ig

    if not edges:
        print("⚠️  No similarity edges — skipping community detection")
        return []

    key_to_idx = {k: i for i, k in enumerate(keys)}
    g = ig.Graph(n=len(keys), directed=False)
    g.vs["store_key"] = keys

    edge_list = []
    weights = []
    for e in edges:
        if e["a"] in key_to_idx and e["b"] in key_to_idx:
            edge_list.append((key_to_idx[e["a"]], key_to_idx[e["b"]]))
            weights.append(e["sim"])
    g.add_edges(edge_list)
    g.es["weight"] = weights

    partition = g.community_leiden(
        weights="weight",
        objective_function="modularity",
        n_iterations=10,
    )

    community_map = {}
    for community_id, members in enumerate(partition):
        for idx in members:
            community_map[keys[idx]] = str(community_id)

    # Write to Neo4j
    rows = [{"store_key": k, "community_id": cid} for k, cid in community_map.items()]
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    SET r.community_id = row.community_id
    WITH r, row
    MERGE (c:Community {community_id: row.community_id})
    SET c.level = $level, c.algorithm = 'igraph.leiden', c.updated_at = datetime()
    MERGE (r)-[:IN_COMMUNITY]->(c)
    """, {"rows": rows, "level": level})

    result = client.run("""
    MATCH (c:Community)<-[:IN_COMMUNITY]-(r:Restaurant)
    RETURN c.community_id AS community_id, count(r) AS size
    ORDER BY size DESC
    """)
    print(f"✅ igraph Leiden: {len(partition)} communities, modularity={partition.modularity:.4f}")
    return result


n_edges, edge_list = build_similarity_edges_from_embeddings(neo4j_client, restaurant_docs)
print("✅ Embedding-based SIMILAR_TO edges:", n_edges)

communities = build_communities_igraph(neo4j_client, restaurant_docs["store_key"].tolist(), edge_list)
print("✅ Communities:")
display(pd.DataFrame(communities).head(10))

def get_restaurant_subgraph(store_key: str, top_reviews: int = 3) -> Dict[str, Any]:
    q = """
    MATCH (r:Restaurant {store_key: $store_key})
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(c:Cuisine)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(g:Category)
    OPTIONAL MATCH (r)-[:HAS_ATMOSPHERE]->(t:AtmosphereTag)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)<-[:ABOUT]-(tu:TextUnit)
    WITH r, a, collect(DISTINCT c.name) AS cuisines, collect(DISTINCT g.name) AS categories,
         collect(DISTINCT t.name) AS atmos, collect(DISTINCT {type: att.type, score: att.score}) AS attrs,
         collect(DISTINCT {text_unit_id: tu.text_unit_id, text: tu.text, sentiment: tu.sentiment, rating: tu.rating})[..$top_reviews] AS text_units
    OPTIONAL MATCH (r)-[s:SIMILAR_TO]-(nbr:Restaurant)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH r, a, cuisines, categories, atmos, attrs, text_units, rep,
         collect(DISTINCT {store_key: nbr.store_key, name: nbr.name, similarity: s.similarity})[..5] AS neighbors
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address, r.gmaps_rating AS rating,
           a.name AS district, a.city AS city, cuisines, categories, atmos, attrs, text_units, neighbors,
           rep.summary AS community_report
    """
    rows = neo4j_client.run(q, {"store_key": store_key, "top_reviews": top_reviews})
    return rows[0] if rows else {}

## 8. LLM-primary intent parsing bằng Pydantic structured output


In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

class RestaurantIntent(BaseModel):
    query_type: Literal["search", "similar", "compare", "personalized"] = "search"
    district: Optional[str] = None
    cuisines: List[str] = Field(default_factory=list)
    categories: List[str] = Field(default_factory=list)
    dish_name: Optional[str] = None
    min_rating: Optional[float] = Field(default=None, ge=0, le=5)
    price_band: Optional[Literal["budget", "mid", "premium"]] = None
    required_attributes: List[Literal["food_quality", "service", "cleanliness", "packaging", "price", "space", "speed"]] = Field(default_factory=list)
    sentiment_pref: Optional[Literal["positive", "neutral", "negative"]] = None
    top_k: int = Field(default=5, ge=1, le=20)

class CommunityReport(BaseModel):
    title: str
    summary: str
    key_strengths: List[str] = Field(default_factory=list)
    cautions: List[str] = Field(default_factory=list)
    representative_restaurants: List[str] = Field(default_factory=list)

class RecommendationAnswer(BaseModel):
    answer: str

def get_llm():
    if LLM_PROVIDER == "anthropic":
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("LLM_PROVIDER=anthropic but ANTHROPIC_API_KEY is missing.")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=ANTHROPIC_API_KEY, temperature=0)
    if LLM_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("LLM_PROVIDER=openai but OPENAI_API_KEY is missing.")
        return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    raise ValueError(f"Unsupported LLM_PROVIDER={LLM_PROVIDER}. Use 'openai' or 'anthropic'.")

llm = get_llm()
intent_llm = llm.with_structured_output(RestaurantIntent)
report_llm = llm.with_structured_output(CommunityReport)
answer_llm = llm.with_structured_output(RecommendationAnswer)

KNOWN_CUISINES = sorted({x for xs in summary["cuisines"] for x in (xs or [])})
KNOWN_CATEGORIES = sorted({x for xs in summary["categories"] for x in (xs or [])})
KNOWN_DISTRICTS = sorted({str(x) for x in summary["district"].dropna().unique().tolist() if str(x).strip()})

INTENT_SYSTEM = """Bạn là bộ phân tích intent cho hệ gợi ý quán ăn GraphRAG.
Trích xuất truy vấn thành schema RestaurantIntent.
Chỉ chọn district/cuisine/category nếu người dùng thật sự nêu hoặc suy ra rõ.
Các required_attributes hợp lệ: food_quality, service, cleanliness, packaging, price, space, speed.
"""
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", INTENT_SYSTEM + "\nDistrict đã biết: {districts}\nCuisine đã biết: {cuisines}\nCategory đã biết: {categories}"),
    ("human", "Query: {query}"),
])

def parse_intent(query: str) -> dict:
    parsed = (intent_prompt | intent_llm).invoke({
        "query": query,
        "districts": KNOWN_DISTRICTS,
        "cuisines": KNOWN_CUISINES,
        "categories": KNOWN_CATEGORIES,
    })
    return parsed.model_dump()


## 9. Graph retrieval + neighborhood / subgraph retrieval

In [ ]:
def graph_candidate_search(intent: dict, top_k: int = 10) -> List[dict]:
    match_lines = ["MATCH (r:Restaurant)"]
    where = []
    params = {"top_k": int(top_k)}
    if intent.get("district"):
        match_lines.append("MATCH (r)-[:IN_AREA]->(area:Area)")
        where.append("toLower(area.name) CONTAINS toLower($district)")
        params["district"] = intent["district"]
    if intent.get("price_band"):
        match_lines.append("MATCH (r)-[:HAS_PRICE_BAND]->(pb:PriceBand)")
        where.append("pb.name = $price_band")
        params["price_band"] = intent["price_band"]
    if intent.get("cuisines"):
        match_lines.append("MATCH (r)-[:HAS_CUISINE]->(cui:Cuisine)")
        where.append("cui.name IN $cuisines")
        params["cuisines"] = intent["cuisines"]
    if intent.get("categories"):
        match_lines.append("MATCH (r)-[:HAS_CATEGORY]->(cat:Category)")
        where.append("cat.name IN $categories")
        params["categories"] = intent["categories"]
    if intent.get("min_rating") is not None:
        where.append("coalesce(r.gmaps_rating, 0) >= $min_rating")
        params["min_rating"] = float(intent["min_rating"])
    for i, attr in enumerate(intent.get("required_attributes", [])):
        alias = f"att{i}"
        match_lines.append(f"MATCH (r)-[:HAS_ATTRIBUTE]->({alias}:Attribute {{type: $attr_{i}}})")
        where.append(f"{alias}.score >= 0.15")
        params[f"attr_{i}"] = attr
    q = "\n".join(match_lines) + "\n"
    if where:
        q += "WHERE " + " AND ".join(where) + "\n"
    q += """
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH r, a, rep, collect(DISTINCT {type: att.type, score: att.score}) AS attributes
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address,
           a.name AS district, a.city AS city, r.gmaps_rating AS rating,
           attributes, rep.summary AS community_report
    ORDER BY r.gmaps_rating DESC NULLS LAST
    LIMIT $top_k
    """
    return neo4j_client.run(q, params)

def subgraph_expand_candidates(seed_store_keys: List[str], max_neighbors: int = 6) -> List[dict]:
    if not seed_store_keys:
        return []
    q = """
    UNWIND $keys AS key
    MATCH (r:Restaurant {store_key: key})-[s:SIMILAR_TO]-(nbr:Restaurant)
    OPTIONAL MATCH (nbr)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (nbr)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (nbr)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH nbr, max(s.similarity) AS sim, a, rep, collect(DISTINCT {type: att.type, score: att.score}) AS attributes
    RETURN nbr.store_key AS store_key, nbr.name AS name, nbr.address AS address,
           a.name AS district, a.city AS city, nbr.gmaps_rating AS rating,
           sim, attributes, rep.summary AS community_report
    ORDER BY sim DESC, rating DESC
    LIMIT $max_neighbors
    """
    return neo4j_client.run(q, {"keys": seed_store_keys, "max_neighbors": max_neighbors})


## 10. Vector retrieval cho Restaurant summary và TextUnit


In [ ]:
def vector_search_restaurants(query: str, top_k: int = 8) -> List[dict]:
    hits = qdrant.search(collection_name=COLL_RESTAURANT, query_vector=emb_query(query), limit=top_k, with_payload=True)
    return [{
        "store_key": h.payload["store_key"], "name": h.payload["name"], "address": h.payload.get("address"),
        "district": h.payload.get("district"), "city": h.payload.get("city"), "rating": h.payload.get("rating"),
        "price_band": h.payload.get("price_band"), "doc_text": h.payload.get("doc_text"),
        "vec_score": round(float(h.score), 4), "source": "restaurant_vector",
    } for h in hits]

def vector_search_text_units(query: str, top_k: int = 16, store_keys: Optional[List[str]] = None) -> List[dict]:
    hits = qdrant.search(collection_name=COLL_TEXT_UNIT, query_vector=emb_query(query), limit=top_k, with_payload=True)
    rows = []
    allow = set(store_keys) if store_keys else None
    for h in hits:
        p = h.payload
        if allow and p.get("store_key") not in allow:
            continue
        rows.append({
            "store_key": p["store_key"], "text_unit_id": p["text_unit_id"], "review_id": p["review_id"],
            "store_name": p.get("store_name"), "rating": p.get("rating"), "sentiment": p.get("sentiment"),
            "feedback": p.get("feedback"), "aspect_scores": p.get("aspect_scores"), "doc_text": p.get("doc_text"),
            "vec_score": round(float(h.score), 4), "source": "text_unit_vector",
        })
    return rows


## 11. Fusion + rerank bằng Reciprocal Rank Fusion


In [ ]:
def aggregate_text_unit_evidence(text_unit_hits: List[dict]) -> Dict[str, dict]:
    bucket = defaultdict(lambda: {"evidence": [], "text_unit_vec_score_max": 0.0})
    for r in text_unit_hits:
        b = bucket[r["store_key"]]
        b["evidence"].append(r["feedback"])
        b["text_unit_vec_score_max"] = max(b["text_unit_vec_score_max"], r.get("vec_score", 0.0))
    return bucket

def _rrf(rank: int, k: int = RRF_K) -> float:
    return 1.0 / (k + rank)

def rerank_candidates(
    query: str,
    intent: dict,
    graph_hits: List[dict],
    neighbor_hits: List[dict],
    rest_vec_hits: List[dict],
    text_unit_hits: List[dict],
) -> List[dict]:
    """FIX (Vấn đề 4): Balanced scoring formula.

    Vấn đề cũ: rating bonus = rating/5 / (RRF_K+1) ≈ 0.016 max
    = xấp xỉ score của rank-1 trong 1 nguồn → rating bị overwhelm.

    Fix: rating bonus được tính riêng theo normalized [0,1] scale
    rồi nhân với RATING_WEIGHT (configurable). Mặc định 0.1 = 10% tổng score,
    đủ để phân biệt 4-star vs 3-star nhưng không dominated over retrieval rank.
    """
    RATING_WEIGHT = 0.10  # rating's share in final score

    by_store: Dict[str, dict] = {}

    def ensure(rec: dict):
        sid = rec["store_key"]
        if sid not in by_store:
            by_store[sid] = {
                "store_key": sid,
                "name": rec.get("name"),
                "address": rec.get("address"),
                "district": rec.get("district"),
                "city": rec.get("city"),
                "rating": rec.get("rating"),
                "rrf_score": 0.0,
                "graph_rank_score": 0.0,
                "neighbor_score": 0.0,
                "restaurant_vec_score": 0.0,
                "text_unit_vec_score": 0.0,
                "community_report": rec.get("community_report"),
                "evidence": [],
                "source_flags": set(),
            }
        else:
            by_store[sid]["community_report"] = by_store[sid].get("community_report") or rec.get("community_report")
        return by_store[sid]

    for rank, r in enumerate(graph_hits, start=1):
        c = ensure(r)
        c["graph_rank_score"] = max(c["graph_rank_score"], _rrf(rank))
        c["rrf_score"] += _rrf(rank)
        c["source_flags"].add("graph_filter")

    for rank, r in enumerate(neighbor_hits, start=1):
        c = ensure(r)
        c["neighbor_score"] = max(c["neighbor_score"], float(r.get("sim", 0.0)))
        c["rrf_score"] += _rrf(rank)
        c["source_flags"].add("graph_neighbor")

    for rank, r in enumerate(rest_vec_hits, start=1):
        c = ensure(r)
        c["restaurant_vec_score"] = max(c["restaurant_vec_score"], float(r.get("vec_score", 0.0)))
        c["rrf_score"] += _rrf(rank)
        c["source_flags"].add("restaurant_vector")

    text_unit_bucket = aggregate_text_unit_evidence(text_unit_hits)
    for sid, info in text_unit_bucket.items():
        c = ensure({"store_key": sid, "name": name_map.get(sid)})
        c["text_unit_vec_score"] = max(c["text_unit_vec_score"], info["text_unit_vec_score_max"])
        c["evidence"] = info["evidence"][:3]
        # text_unit RRF contribution (score-based, not rank-based; treat as one extra source)
        c["rrf_score"] += _rrf(1) * info["text_unit_vec_score_max"]
        c["source_flags"].add("text_unit_vector")

    # Normalize RRF scores to [0, 1] range across all candidates, then blend with rating
    rrf_scores = [c["rrf_score"] for c in by_store.values()]
    max_rrf = max(rrf_scores) if rrf_scores else 1.0
    max_rrf = max(max_rrf, 1e-9)

    results = []
    for c in by_store.values():
        rating = float(c["rating"] or 0.0)
        rrf_norm = c["rrf_score"] / max_rrf  # [0, 1]
        rating_norm = rating / 5.0           # [0, 1]
        # FIX: balanced blend — retrieval dominates (90%), rating secondary (10%)
        c["final_score"] = (1.0 - RATING_WEIGHT) * rrf_norm + RATING_WEIGHT * rating_norm
        c["source_flags"] = sorted(c["source_flags"])
        results.append(c)

    results.sort(key=lambda x: x["final_score"], reverse=True)
    return results


def hybrid_retrieve(query: str, top_k: int = 5) -> Tuple[dict, List[dict]]:
    intent = parse_intent(query)
    graph_hits = graph_candidate_search(intent, top_k=max(10, top_k))
    rest_vec_hits = vector_search_restaurants(query, top_k=max(10, top_k))
    seed_keys = list(dict.fromkeys([r["store_key"] for r in graph_hits[:3]] + [r["store_key"] for r in rest_vec_hits[:3]]))
    neighbor_hits = subgraph_expand_candidates(seed_keys, max_neighbors=8)
    store_scope = list({*(r["store_key"] for r in graph_hits), *(r["store_key"] for r in rest_vec_hits), *(r["store_key"] for r in neighbor_hits)})
    text_unit_hits = vector_search_text_units(query, top_k=20, store_keys=store_scope if store_scope else None)
    ranked = rerank_candidates(query, intent, graph_hits, neighbor_hits, rest_vec_hits, text_unit_hits)
    return intent, ranked[:top_k]

## 11.1. Cross-encoder reranking (BGE-Reranker)

In [ ]:
# FIX (Vấn đề 5): Cross-encoder reranking sau RRF.
# Bi-encoder tốt cho recall, cross-encoder tốt cho precision ở top-k.
# Dùng BAAI/bge-reranker-base (hỗ trợ Vietnamese tốt, nhẹ ~400MB).

try:
    from FlagEmbedding import FlagReranker
    _RERANKER_AVAILABLE = True
except ImportError:
    _RERANKER_AVAILABLE = False
    print("⚠️  FlagEmbedding not installed. Cross-encoder reranking disabled. Run: pip install FlagEmbedding")

CROSS_ENCODER_MODEL = os.getenv("CROSS_ENCODER_MODEL", "BAAI/bge-reranker-base")
_reranker = None

def get_reranker():
    global _reranker
    if not _RERANKER_AVAILABLE:
        return None
    if _reranker is None:
        print(f"Loading cross-encoder: {CROSS_ENCODER_MODEL}...")
        _reranker = FlagReranker(CROSS_ENCODER_MODEL, use_fp16=torch.cuda.is_available())
        print("✅ Cross-encoder loaded")
    return _reranker


def cross_encoder_rerank(
    query: str,
    candidates: List[dict],
    top_k: int = 5,
    ce_weight: float = 0.3,
) -> List[dict]:
    """FIX (Vấn đề 5): Rerank top candidates bằng cross-encoder.

    Dùng community_report + doc summary làm passage cho cross-encoder.
    ce_weight: blending weight với RRF final_score (default 30% CE, 70% RRF).
    Nếu reranker không available, trả về candidates gốc không thay đổi.
    """
    reranker = get_reranker()
    if reranker is None or not candidates:
        return candidates[:top_k]

    passages = []
    for c in candidates:
        passage = " | ".join(filter(None, [
            c.get("name", ""),
            c.get("address", ""),
            c.get("community_report", ""),
            " ".join((c.get("evidence") or [])[:2]),
        ]))
        passages.append(passage[:512])

    pairs = [[query, p] for p in passages]
    ce_scores = reranker.compute_score(pairs, normalize=True)  # [0,1]

    # Blend: final_score already in [0,1] from normalized RRF
    for i, c in enumerate(candidates):
        rrf_s = c.get("final_score", 0.0)
        ce_s = float(ce_scores[i]) if isinstance(ce_scores, list) else float(ce_scores)
        c["ce_score"] = round(ce_s, 4)
        c["final_score_before_ce"] = rrf_s
        c["final_score"] = (1.0 - ce_weight) * rrf_s + ce_weight * ce_s

    candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates[:top_k]


# Patch hybrid_retrieve to include cross-encoder step
_hybrid_retrieve_rrf = hybrid_retrieve  # keep reference

def hybrid_retrieve(query: str, top_k: int = 5, use_cross_encoder: bool = True) -> Tuple[dict, List[dict]]:
    """Full retrieval pipeline: intent → graph → vector → RRF → cross-encoder."""
    intent, ranked_rrf = _hybrid_retrieve_rrf(query, top_k=max(top_k * 2, 10))
    if use_cross_encoder:
        ranked = cross_encoder_rerank(query, ranked_rrf, top_k=top_k)
    else:
        ranked = ranked_rrf[:top_k]
    return intent, ranked

print("✅ Cross-encoder reranker module ready (lazy-loads on first use)")

## 12. Community reports + generation/recommendation


In [ ]:
# FIX (Vấn đề 9): Community report prompt mở rộng với anti-hallucination,
# hierarchy guidance, format rõ ràng — tăng chất lượng global query.
REPORT_SYSTEM = """Bạn là chuyên gia phân tích hệ thống quán ăn sử dụng phương pháp GraphRAG.
Nhiệm vụ: Viết CommunityReport cho một cụm (community) nhà hàng được phát hiện tự động.

== YÊU CẦU BẮT BUỘC ==
1. Chỉ dùng thông tin từ context cung cấp. Không được suy diễn ngoài context.
2. title: tên ngắn gọn mô tả đặc trưng cụm (ví dụ: "Cụm quán bún bò Hoàng Mai rating cao")
3. summary: 3-5 câu mô tả đặc điểm chung, khu vực địa lý, phân khúc giá, điểm nổi bật.
4. key_strengths: 2-5 điểm mạnh cụ thể với dẫn chứng từ aspect scores hoặc review.
   Ví dụ: "food_quality trung bình +0.72 — phần lớn review khen đồ ăn ngon"
5. cautions: 1-3 điểm yếu hoặc thông tin cần lưu ý (nếu context không có, để list rỗng).
6. representative_restaurants: 2-5 tên quán tiêu biểu nhất trong cụm.

== CHẤT LƯỢNG ==
- Không viết câu chung chung như "quán có không khí tốt" nếu không có evidence.
- Không lặp lại tên quán trong summary.
- Nếu cụm nhỏ (< 3 quán): ghi rõ "Cụm nhỏ, kết quả có thể không đại diện."
- Nếu context thiếu dữ liệu: ghi rõ "Dữ liệu aspect không đầy đủ" trong cautions.
- Không bịa tên món ăn, địa điểm hoặc số liệu không có trong context.

== FORMAT ==
Trả về JSON theo Pydantic schema CommunityReport.
"""

report_prompt = ChatPromptTemplate.from_messages([
    ("system", REPORT_SYSTEM),
    ("human", "Community id: {community_id}\n\nContext (JSON list of restaurants in this community):\n{context}\n\nViết CommunityReport cho community trên."),
])

def build_community_context(community_id: str, limit: int = 30) -> str:
    rows = neo4j_client.run("""
    MATCH (c:Community {community_id: $community_id})<-[:IN_COMMUNITY]-(r:Restaurant)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(a:Attribute)
    OPTIONAL MATCH (r)<-[:ABOUT]-(tu:TextUnit)
    WITH r,
         collect(DISTINCT {type:a.type, score:a.score}) AS attrs,
         collect(DISTINCT tu.text)[..3] AS text_units
    RETURN r.name AS name, r.address AS address, r.gmaps_rating AS rating,
           r.district AS district, r.city AS city,
           attrs, text_units
    LIMIT $limit
    """, {"community_id": str(community_id), "limit": limit})
    return json.dumps(rows, ensure_ascii=False, indent=2)

def upsert_community_reports():
    communities_list = neo4j_client.run(
        "MATCH (c:Community)<-[:IN_COMMUNITY]-(:Restaurant) RETURN c.community_id AS community_id ORDER BY community_id"
    )
    rows = []
    for c in tqdm(communities_list, desc="Community reports"):
        cid = str(c["community_id"])
        context = build_community_context(cid)
        if not context or context == "[]":
            print(f"  ⚠️  Community {cid}: empty context, skipping")
            continue
        try:
            report = (report_prompt | report_llm).invoke({"community_id": cid, "context": context})
            rows.append({
                "community_id": cid,
                "report_id": f"community_report_{cid}",
                "title": report.title,
                "summary": report.summary,
                "key_strengths": report.key_strengths,
                "cautions": report.cautions,
                "representative_restaurants": report.representative_restaurants,
            })
        except Exception as e:
            print(f"  ⚠️  Community {cid} report failed: {e}")
    neo4j_client.run("""
    UNWIND $rows AS row
    MATCH (c:Community {community_id: row.community_id})
    MERGE (cr:CommunityReport {report_id: row.report_id})
    SET cr.title = row.title, cr.summary = row.summary, cr.key_strengths = row.key_strengths,
        cr.cautions = row.cautions, cr.representative_restaurants = row.representative_restaurants,
        cr.updated_at = datetime()
    MERGE (c)-[:HAS_REPORT]->(cr)
    """, {"rows": rows})
    print(f"✅ Upserted community reports: {len(rows)}")

upsert_community_reports()

RECOMMEND_SYSTEM = """Bạn là trợ lý gợi ý quán ăn.
Dựa duy nhất trên context đã retrieval/rerank:
- nêu top quán phù hợp nhất
- giải thích vì sao phù hợp
- trích dẫn ngắn evidence review nếu có
- nếu context thiếu, nói rõ thiếu dữ liệu, không tự bịa.
"""
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system", RECOMMEND_SYSTEM),
    ("human", "Query: {query}\nIntent: {intent}\n\nContext:\n{context}"),
])

def format_retrieval_context(rows: List[dict]) -> str:
    lines = []
    for i, r in enumerate(rows, 1):
        ev = " | ".join(r.get("evidence", [])[:2]) if r.get("evidence") else ""
        ce = f" | ce_score={r['ce_score']:.4f}" if r.get("ce_score") is not None else ""
        lines.append(
            f"{i}. {r.get('name')} | rating={r.get('rating')} | district={r.get('district')} | "
            f"score={r.get('final_score'):.4f}{ce} | sources={','.join(r.get('source_flags', []))}\n"
            f"   address={r.get('address')}\n"
            f"   community_report={r.get('community_report') or ''}\n"
            f"   evidence={ev}"
        )
    return "\n\n".join(lines)

def recommend(query: str, top_k: int = 5) -> str:
    intent, ranked = hybrid_retrieve(query, top_k=top_k)
    context = format_retrieval_context(ranked)
    answer = (recommend_prompt | answer_llm).invoke({
        "query": query,
        "intent": json.dumps(intent, ensure_ascii=False),
        "context": context,
    })
    return answer.answer

## 13. Evaluation cho retrieval quality

In [ ]:
def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant); ret = retrieved[:k]
    return len(rel.intersection(ret)) / len(rel) if rel else 0.0

def mrr_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    for i, sid in enumerate(retrieved[:k], start=1):
        if sid in rel:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    dcg = 0.0
    for i, sid in enumerate(retrieved[:k], start=1):
        gain = 1.0 if sid in rel else 0.0
        dcg += gain / math.log2(i + 1)
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def generate_eval_dataset_from_graph(
    n_queries: int = 20,
    seed: int = 42,
) -> List[dict]:
    """FIX (Vấn đề 8): Auto-generate ground truth eval dataset bằng LLM từ graph.

    Strategy:
    1. Sample n_queries community reports hoặc restaurant clusters từ Neo4j
    2. Gọi LLM sinh câu hỏi tự nhiên dựa trên context của cluster
    3. Dùng restaurants trong cluster làm relevant_store_keys (silver labels)

    Đây là "silver standard" — không phải human labels, nhưng đủ để:
    - So sánh các variant (vector-only vs graph vs hybrid)
    - Detect regression khi thay đổi pipeline
    """
    GEN_SYSTEM = """Bạn tạo câu hỏi tìm kiếm quán ăn tự nhiên bằng tiếng Việt dựa trên thông tin về một nhóm nhà hàng.
Câu hỏi phải tự nhiên như người dùng thật hỏi (không quá kỹ thuật).
Trả về JSON: {"query": "...", "reasoning": "tại sao nhà hàng này relevant"}
Chỉ trả về JSON, không giải thích thêm."""

    rows = neo4j_client.run("""
    MATCH (c:Community)-[:HAS_REPORT]->(cr:CommunityReport)
    MATCH (c)<-[:IN_COMMUNITY]-(r:Restaurant)
    WITH c.community_id AS cid, cr.summary AS summary, cr.title AS title,
         collect(r.store_key) AS store_keys, collect(r.name) AS names, count(r) AS n
    WHERE n >= 2
    RETURN cid, summary, title, store_keys, names
    ORDER BY rand()
    LIMIT $n
    """, {"n": n_queries})

    if not rows:
        print("⚠️  No community reports found. Run upsert_community_reports() first.")
        return []

    test_cases = []
    gen_llm_raw = get_llm()
    for row in tqdm(rows, desc="Generating eval queries"):
        context = f"Community: {row['title']}\nRestaurants: {', '.join(row['names'][:5])}\nSummary: {row['summary']}"
        try:
            resp = gen_llm_raw.invoke([
                {"role": "system", "content": GEN_SYSTEM},
                {"role": "user", "content": f"Context:\n{context}\n\nSinh 1 câu hỏi tìm quán ăn phù hợp với cụm này."},
            ])
            raw = resp.content.strip()
            raw = re.sub(r"```json|```", "", raw).strip()
            parsed = json.loads(raw)
            test_cases.append({
                "query": parsed["query"],
                "relevant_store_keys": row["store_keys"],
                "community_id": row["cid"],
                "reasoning": parsed.get("reasoning", ""),
            })
        except Exception as e:
            print(f"  ⚠️  Failed to generate query for community {row['cid']}: {e}")

    print(f"✅ Generated {len(test_cases)} eval queries from graph")
    return test_cases


def evaluate_retrieval(
    test_cases: Optional[List[dict]] = None,
    k: int = 5,
    auto_generate: bool = True,
    n_auto_queries: int = 20,
) -> pd.DataFrame:
    """Evaluate retrieval pipeline.

    Args:
        test_cases: User-provided labeled queries [{query, relevant_store_keys}].
                    If None and auto_generate=True, generates from graph automatically.
        auto_generate: If True and test_cases is None, auto-generate from community reports.
        n_auto_queries: Number of queries to auto-generate.
    """
    if test_cases is None or len(test_cases) == 0:
        if auto_generate:
            print("No test_cases provided. Auto-generating from community reports...")
            test_cases = generate_eval_dataset_from_graph(n_queries=n_auto_queries)
        else:
            raise ValueError(
                "test_cases is empty and auto_generate=False. "
                "Provide labeled queries or set auto_generate=True."
            )
    if not test_cases:
        raise ValueError("Could not generate test cases. Check that community reports exist.")

    rows = []
    for tc in tqdm(test_cases, desc="Evaluating"):
        _, ranked = hybrid_retrieve(tc["query"], top_k=k)
        retrieved = [r["store_key"] for r in ranked]
        rows.append({
            "query": tc["query"],
            "relevant": tc["relevant_store_keys"],
            f"recall@{k}": recall_at_k(retrieved, tc["relevant_store_keys"], k),
            f"mrr@{k}": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
            f"ndcg@{k}": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            "retrieved": retrieved,
        })
    df = pd.DataFrame(rows)
    print("\n=== Evaluation Results ===")
    print(df[[f"recall@{k}", f"mrr@{k}", f"ndcg@{k}"]].mean().round(4).to_string())
    return df


# --- Ablation using evaluate_retrieval ---
def evaluate_ablation(test_cases: Optional[List[dict]] = None, k: int = 5) -> pd.DataFrame:
    """So sánh 4 variants: vector-only, graph-only, text-unit-only, hybrid."""
    if test_cases is None:
        test_cases = generate_eval_dataset_from_graph(n_queries=15)

    def _eval_fn(retrieve_fn):
        results = []
        for tc in test_cases:
            retrieved = retrieve_fn(tc["query"], top_k=k)
            results.append({
                "recall": recall_at_k(retrieved, tc["relevant_store_keys"], k),
                "mrr": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
                "ndcg": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            })
        return pd.DataFrame(results).mean()

    variants = {
        "vector_only": lambda q, top_k: [r["store_key"] for r in vector_search_restaurants(q, top_k=top_k)],
        "graph_only": lambda q, top_k: [r["store_key"] for r in graph_candidate_search(parse_intent(q), top_k=top_k)],
        "text_unit_only": lambda q, top_k: [r["store_key"] for r in vector_search_text_units(q, top_k=top_k)],
        "hybrid_no_ce": lambda q, top_k: retrieve_hybrid(q, top_k=top_k),
        "hybrid_with_ce": lambda q, top_k: [r["store_key"] for r in hybrid_retrieve(q, top_k=top_k, use_cross_encoder=True)[1]],
    }

    records = []
    for name, fn in variants.items():
        print(f"Evaluating: {name}...")
        scores = _eval_fn(fn)
        records.append({"variant": name, **scores.to_dict()})
    df = pd.DataFrame(records).set_index("variant")
    print("\n=== Ablation Results ===")
    print(df.round(4).to_string())
    return df

## 14. Ablation gợi ý

In [ ]:
def retrieve_vector_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_restaurants(query, top_k=top_k)]

def retrieve_graph_only(query: str, top_k: int = 5) -> List[str]:
    intent = parse_intent(query)
    return [r["store_key"] for r in graph_candidate_search(intent, top_k=top_k)]

def retrieve_text_unit_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_text_units(query, top_k=top_k)]

def retrieve_hybrid(query: str, top_k: int = 5) -> List[str]:
    _, ranked = hybrid_retrieve(query, top_k=top_k)
    return [r["store_key"] for r in ranked]
